# TOFU official-metric baseline retry: isolated 8B retain90 oracle

This version restores checksum-verified 3B full/oracle outputs and the completed 8B full output, then evaluates only the missing 8B retain90 oracle in a fresh Kaggle runtime. This avoids reloading and rerunning the 70-minute 8B full evaluation in the same process. It disables only the pinned Transformers 5.22.1 load-time allocator preallocation for 8B, places weights with `balanced` across both T4s under a 13 GiB-per-GPU cap, and refuses scoring if any 8B weight is on CPU/disk. A four-token generation smoke must pass first.

The run is inference-only: it does not rerun full checkpoints, train NPO, or create a research adapter. Kaggle T4 inference remains FP16 + SDPA; this differs from the upstream BF16 + FlashAttention 2 runtime.

In [ ]:
import importlib.metadata as metadata
import subprocess, sys
required = {'transformers': '5.22.1', 'accelerate': '1.15.0', 'scipy': '1.17.1', 'rouge-score': '0.1.2'}
missing = []
for package, wanted in required.items():
    try: found = metadata.version(package)
    except metadata.PackageNotFoundError: found = None
    if found != wanted: missing.append(f'{package}=={wanted}')
if missing: subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *missing], check=True)
print({p: metadata.version(p) for p in required})

In [ ]:
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError('Select GPU T4 x2 before starting; stopped before loading checkpoint weights.')
gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print({'gpus': gpus, 'memory_bytes': [torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())]})
if gpus.count('Tesla T4') < 2: raise RuntimeError(f'Expected two Tesla T4 GPUs, found {gpus}')

In [ ]:
%%writefile /kaggle/working/diagnose_tofu.py
#!/usr/bin/env python3
"""Audit pinned TOFU data and pinned Open-Unlearning checkpoints without loading weights."""

from __future__ import annotations

import argparse
import collections
import hashlib
import json
import platform
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


DATASET_ID = "locuslab/TOFU"
DATASET_REVISION = "324592d84ae4f482ac7249b9285c2ecdb53e3a68"
OPEN_UNLEARNING_REVISION = "2d98027677f3468d52b6baa3fc4f2b9f2dc08820"
DATA_FILES = (
    "full.json",
    "forget10.json",
    "retain90.json",
    "holdout10.json",
    "forget10_perturbed.json",
    "retain_perturbed.json",
    "real_authors.json",
    "real_authors_perturbed.json",
    "world_facts.json",
    "world_facts_perturbed.json",
)
EXPECTED_ROWS = {
    "full.json": 4000,
    "forget10.json": 400,
    "retain90.json": 3600,
    "holdout10.json": 400,
    "forget10_perturbed.json": 400,
    "retain_perturbed.json": 400,
    "real_authors.json": 100,
    "real_authors_perturbed.json": 100,
    "world_facts.json": 117,
    "world_facts_perturbed.json": 117,
}
PROFILE_QA_COUNT = 20

CHECKPOINTS = {
    "llama_3_2_3b_full": (
        "open-unlearning/tofu_Llama-3.2-3B-Instruct_full",
        "24f31ca19f6966dcb6f6b29abc511cce71222d4a",
    ),
    "llama_3_2_3b_retain90": (
        "open-unlearning/tofu_Llama-3.2-3B-Instruct_retain90",
        "2c35819e9a731d096df9abd6efbcc361870c5333",
    ),
    "llama_3_1_8b_full": (
        "open-unlearning/tofu_Llama-3.1-8B-Instruct_full",
        "1a5c5b1a557f8c99bdadecd5168ebd03f640b00e",
    ),
    "llama_3_1_8b_retain90": (
        "open-unlearning/tofu_Llama-3.1-8B-Instruct_retain90",
        "f02956a9f25a18f38ad7808fbefb2f957347273e",
    ),
}


def get_bytes(url: str, timeout: int = 45, attempts: int = 3) -> bytes:
    request = Request(url, headers={"User-Agent": "fsds-tofu-audit/1.0"})
    last_error: Exception | None = None
    for attempt in range(attempts):
        try:
            with urlopen(request, timeout=timeout) as response:
                return response.read()
        except (HTTPError, URLError, TimeoutError) as error:
            last_error = error
            if isinstance(error, HTTPError) and error.code not in (429, 500, 502, 503, 504):
                break
            if attempt + 1 < attempts:
                time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"GET failed for {url}: {last_error}")


def load_jsonl(raw: bytes, name: str) -> list[dict]:
    rows: list[dict] = []
    for line_number, line in enumerate(raw.decode("utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError as error:
            raise ValueError(f"{name}:{line_number}: invalid JSON: {error}") from error
        if not isinstance(row, dict):
            raise ValueError(f"{name}:{line_number}: expected a JSON object")
        rows.append(row)
    return rows


def duplicate_summary(rows: list[dict]) -> dict:
    qa_counts = collections.Counter((row.get("question"), row.get("answer")) for row in rows)
    question_answers: dict[str, set[str]] = collections.defaultdict(set)
    question_counts = collections.Counter()
    for row in rows:
        question = row.get("question")
        answer = row.get("answer")
        if isinstance(question, str):
            question_counts[question] += 1
            question_answers[question].add(answer)
    repeated_questions = [
        {"question": question, "count": count, "answers": sorted(question_answers[question])}
        for question, count in question_counts.items()
        if count > 1
    ]
    return {
        "exact_duplicate_qa_rows": sum(count - 1 for count in qa_counts.values()),
        "repeated_question_rows": sum(count - 1 for count in question_counts.values()),
        "repeated_question_strings": len(repeated_questions),
        "conflicting_answer_questions": sum(len(item["answers"]) > 1 for item in repeated_questions),
        "repeated_questions": repeated_questions,
    }


def audit_data() -> dict:
    rows_by_file: dict[str, list[dict]] = {}
    file_info: dict[str, dict] = {}
    for name in DATA_FILES:
        url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/{name}"
        raw = get_bytes(url)
        rows = load_jsonl(raw, name)
        rows_by_file[name] = rows
        file_info[name] = {
            "url": url,
            "sha256": hashlib.sha256(raw).hexdigest(),
            "bytes": len(raw),
            "rows": len(rows),
            "expected_rows": EXPECTED_ROWS[name],
            "required_fields": ["question", "answer"],
            "missing_or_empty_required_fields": sum(
                1
                for row in rows
                if not all(isinstance(row.get(key), str) and row[key].strip() for key in ("question", "answer"))
            ),
            "duplicates": duplicate_summary(rows),
        }

    base_splits = ("forget10.json", "retain90.json", "holdout10.json")
    qa_sets = {
        name: {(row["question"], row["answer"]) for row in rows_by_file[name]}
        for name in base_splits
    }
    question_sets = {
        name: {row["question"] for row in rows_by_file[name]}
        for name in base_splits
    }
    intersections = {}
    for i, left in enumerate(base_splits):
        for right in base_splits[i + 1 :]:
            key = f"{left.removesuffix('.json')}__{right.removesuffix('.json')}"
            intersections[key] = {
                "exact_qa_pairs": len(qa_sets[left] & qa_sets[right]),
                "question_strings": len(question_sets[left] & question_sets[right]),
            }

    full_counter = collections.Counter(
        (row["question"], row["answer"]) for row in rows_by_file["full.json"]
    )
    forget_retain_counter = collections.Counter(
        (row["question"], row["answer"])
        for name in ("forget10.json", "retain90.json")
        for row in rows_by_file[name]
    )

    aligned_pairs = {
        "forget10_perturbed_matches_forget10_in_order": (
            "forget10.json",
            "forget10_perturbed.json",
            5,
            True,
            True,
        ),
        "real_authors_perturbed_matches_source_in_order": (
            "real_authors.json",
            "real_authors_perturbed.json",
            3,
            False,
            False,
        ),
        "world_facts_perturbed_matches_source_in_order": (
            "world_facts.json",
            "world_facts_perturbed.json",
            3,
            False,
            False,
        ),
    }
    alignment_checks = {}
    for check_name, (source_name, perturbed_name, wrong_count, has_paraphrased_question, has_paraphrased_answer) in aligned_pairs.items():
        source_rows = rows_by_file[source_name]
        perturbed_rows = rows_by_file[perturbed_name]
        aligned = len(source_rows) == len(perturbed_rows) and all(
            base.get("question") == pert.get("question") and base.get("answer") == pert.get("answer")
            for base, pert in zip(source_rows, perturbed_rows)
        )
        fields_valid = all(
            (
                not has_paraphrased_answer
                or (isinstance(row.get("paraphrased_answer"), str) and bool(row["paraphrased_answer"].strip()))
            )
            and isinstance(row.get("perturbed_answer"), list)
            and len(row["perturbed_answer"]) == wrong_count
            and all(isinstance(answer, str) and answer.strip() for answer in row["perturbed_answer"])
            and (
                not has_paraphrased_question
                or (isinstance(row.get("paraphrased_question"), str) and bool(row["paraphrased_question"].strip()))
            )
            for row in perturbed_rows
        )
        alignment_checks[check_name] = aligned
        alignment_checks[f"{perturbed_name}_answer_variants_valid"] = fields_valid

    retain_rows = rows_by_file["retain90.json"]
    retain_pairs = collections.defaultdict(list)
    for row_index, row in enumerate(retain_rows):
        retain_pairs[(row["question"], row["answer"])].append(row_index)
    retain_perturbed = rows_by_file["retain_perturbed.json"]
    retain_perturbed_aligned = all(
        len(retain_pairs.get((row["question"], row["answer"]), [])) == 1
        for row in retain_perturbed
    )
    retain_perturbation_fields_valid = all(
        isinstance(row.get("paraphrased_answer"), str)
        and bool(row["paraphrased_answer"].strip())
        and isinstance(row.get("perturbed_answer"), list)
        and len(row["perturbed_answer"]) == 5
        and all(isinstance(answer, str) and answer.strip() for answer in row["perturbed_answer"])
        for row in retain_perturbed
    )
    retain_source_indices = [
        retain_pairs[(row["question"], row["answer"])][0]
        for row in retain_perturbed
        if len(retain_pairs.get((row["question"], row["answer"]), [])) == 1
    ]
    retain_profile_blocks = collections.Counter(index // PROFILE_QA_COUNT for index in retain_source_indices)
    retain_eval_profile_blocks_valid = (
        len(retain_source_indices) == len(set(retain_source_indices))
        and sorted(retain_source_indices) == list(range(20 * PROFILE_QA_COUNT))
        and set(retain_profile_blocks) == set(range(20))
        and all(count == PROFILE_QA_COUNT for count in retain_profile_blocks.values())
    )
    alignment_checks["retain_perturbed_rows_map_uniquely_to_retain90"] = retain_perturbed_aligned
    alignment_checks["retain_perturbed_answer_variants_valid"] = retain_perturbation_fields_valid
    alignment_checks["retain_perturbed_is_20_complete_profile_blocks"] = retain_eval_profile_blocks_valid

    perturbation_checks_pass = all(alignment_checks.values())

    count_checks = all(file_info[name]["rows"] == file_info[name]["expected_rows"] for name in DATA_FILES)
    schema_checks = all(file_info[name]["missing_or_empty_required_fields"] == 0 for name in DATA_FILES)
    disjoint_checks = all(
        item["exact_qa_pairs"] == 0 and item["question_strings"] == 0
        for item in intersections.values()
    )
    partition_check = full_counter == forget_retain_counter
    structural_checks_pass = count_checks and schema_checks and disjoint_checks and partition_check and perturbation_checks_pass

    return {
        "dataset_id": DATASET_ID,
        "dataset_revision": DATASET_REVISION,
        "files": file_info,
        "split_intersections": intersections,
        "checks": {
            "expected_record_counts": count_checks,
            "required_schema_and_nonempty_text": schema_checks,
            "forget_retain_holdout_pairwise_disjoint_by_question_and_answer": disjoint_checks,
            "full_equals_forget10_plus_retain90_as_multiset": partition_check,
            **alignment_checks,
            "all_official_eval_and_perturbation_alignment_checks_pass": perturbation_checks_pass,
            "structural_checks_pass": structural_checks_pass,
        },
        "canonical_retain_eval": {
            "file": "retain_perturbed.json",
            "rows": len(retain_perturbed),
            "source_profile_blocks_in_retain90": sorted(retain_profile_blocks),
            "source_row_indices_in_retain90": retain_source_indices,
        },
        "profile_grouping": {
            "published_question_answer_pairs_per_profile": PROFILE_QA_COUNT,
            "inferred_profile_counts_from_split_sizes": {
                name.removesuffix(".json"): len(rows_by_file[name]) // PROFILE_QA_COUNT
                for name in ("forget10.json", "retain90.json", "holdout10.json")
            },
            "author_id_column_present": False,
            "limitation": "The release has question/answer text but no author_id field; author isolation is the benchmark's documented construction, not independently proven from a structured ID column here.",
        },
    }


def audit_checkpoints() -> dict:
    results = {}
    for label, (repo, revision) in CHECKPOINTS.items():
        tree_url = f"https://huggingface.co/api/models/{repo}/tree/{revision}?recursive=true&expand=true"
        files = json.loads(get_bytes(tree_url))
        config_raw = get_bytes(f"https://huggingface.co/{repo}/resolve/{revision}/config.json")
        config = json.loads(config_raw)
        weight_files = [item for item in files if item.get("path", "").endswith(".safetensors")]
        available_paths = {item.get("path") for item in files}
        required_tokenizer_files = {"tokenizer.json", "tokenizer_config.json", "special_tokens_map.json"}
        tokenizer_files_present = required_tokenizer_files <= available_paths
        results[label] = {
            "repo_id": repo,
            "revision": revision,
            "config_http_access": "PASS",
            "config_sha256": hashlib.sha256(config_raw).hexdigest(),
            "model_type": config.get("model_type"),
            "hidden_size": config.get("hidden_size"),
            "num_hidden_layers": config.get("num_hidden_layers"),
            "weight_file_count": len(weight_files),
            "weight_bytes_from_hub_metadata": sum(int(item["size"]) for item in weight_files),
            "required_tokenizer_files": sorted(required_tokenizer_files),
            "tokenizer_files_available": tokenizer_files_present,
            "weights_downloaded": False,
        }
        results[label]["checkpoint_access_checks_pass"] = (
            results[label]["config_http_access"] == "PASS"
            and tokenizer_files_present
            and bool(weight_files)
            and all(int(item.get("size", 0)) > 0 for item in weight_files)
        )
    return results


def runtime_context() -> dict:
    try:
        import torch

        return {
            "platform": platform.platform(),
            "torch_version": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "cuda_device_count": torch.cuda.device_count(),
            "mps_available": bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()),
        }
    except Exception as error:  # Runtime metadata is informative, not a data gate.
        return {"platform": platform.platform(), "torch_probe_error": f"{type(error).__name__}: {error}"}


def build_markdown(report: dict) -> str:
    lines = [
        "# TOFU 3B/8B readiness diagnostic",
        "",
        f"Generated: {report['generated_at_utc']}",
        "",
        "## Data integrity",
        "",
        f"Status: **{'PASS' if report['data_audit']['checks']['structural_checks_pass'] else 'FAIL'}; one known data ambiguity is reported below**",
        "",
        "| File | Rows | SHA-256 | Empty required fields | Exact duplicate QA rows | Repeated questions |",
        "|---|---:|---|---:|---:|---:|",
    ]
    for name, info in report["data_audit"]["files"].items():
        dup = info["duplicates"]
        lines.append(
            f"| `{name}` | {info['rows']:,} | `{info['sha256']}` | {info['missing_or_empty_required_fields']} | {dup['exact_duplicate_qa_rows']} | {dup['repeated_question_strings']} |"
        )
    lines += [
        "",
        "Forget10 and retain90 form the full 4,000-row training set exactly. Forget10, retain90, and holdout10 have no exact question overlap with one another. The official evaluation perturbation files are checked against their source rows, field schemas, and expected wrong-answer counts. The 400-row `retain_perturbed` evaluation set maps exactly to the first 20 complete retain90 profile blocks; the evaluator must use this canonical set rather than substitute a custom sample.",
        "",
        "### Data warning",
        "",
    ]
    retain_duplicates = report["data_audit"]["files"]["retain90.json"]["duplicates"]["repeated_questions"]
    if retain_duplicates:
        for item in retain_duplicates:
            answer_text = " / ".join(item["answers"])
            lines.append(
                f"- `retain90` contains the same question {item['count']} times with {len(item['answers'])} different answers: {item['question']} ({answer_text}). This is one ambiguous QA collision; preserve the official split for comparability and report a sensitivity analysis excluding both rows."
            )
    else:
        lines.append("- No repeated question strings found.")
    pg = report["data_audit"]["profile_grouping"]
    counts = pg["inferred_profile_counts_from_split_sizes"]
    lines += [
        "",
        "The published design gives 20 QA pairs per fictitious author. Split sizes therefore correspond to 20 forget profiles, 180 retain profiles, and 20 holdout profiles. The released JSON has no explicit author ID, so this audit does not claim an independent structured-ID leakage proof.",
        "",
        "## Checkpoint availability and resource facts",
        "",
        "| Checkpoint | Immutable revision | Weight shards | Weight size (decimal GB) | Config | Tokenizer files | Weights downloaded? |",
        "|---|---|---:|---:|---|---|",
    ]
    for label, item in report["checkpoints"].items():
        model = "Llama 3.2 3B" if "3b" in label else "Llama 3.1 8B"
        role = "full" if label.endswith("_full") else "retain90 oracle"
        size_gb = item["weight_bytes_from_hub_metadata"] / 1e9
        lines.append(
            f"| {model} {role} | `{item['revision']}` | {item['weight_file_count']} | {size_gb:.2f} | {item['config_http_access']} | {'PASS' if item['tokenizer_files_available'] else 'FAIL'} | no |"
        )
    rt = report["local_runtime"]
    lines += [
        "",
        f"Local runtime: `{rt.get('platform')}`; CUDA available: `{rt.get('cuda_available', 'unknown')}`; CUDA devices: `{rt.get('cuda_device_count', 'unknown')}`; MPS available: `{rt.get('mps_available', 'unknown')}`.",
        "",
        "An earlier Kaggle T4×2 smoke passed one synthetic 8B LoRA step and one synthetic NPO update. That verifies only the synthetic runtime path; it is not a TOFU score or an unlearning result. This local diagnostic did not download model weights or start training.",
        "",
        "## Interpretation and next gate",
        "",
        "TOFU is a workable public benchmark for a controlled pilot: the file-level splits are internally consistent and disjoint, and published 3B/8B full and retain90 checkpoints are reachable by immutable revision. The repeated generic retain question is a small but real label ambiguity. The author-group structure has no explicit IDs in the downloaded schema, which limits local leakage verification.",
        "",
        "TOFU uses synthetic fictional author profiles. Results on it do not establish deletion of real personal data or real-world privacy protection.",
        "",
        "An earlier Kaggle run measured answer likelihood on 1,200 examples per checkpoint, but it used a custom retain sample and did not include generation, Truth Ratio, model utility, or the retain-only oracle comparison; treat it as exploratory. The new pinned official-metric evaluator is separate and has not yet run. Run it on Kaggle before NPO, archive all question-level outputs and per-profile uncertainty, then review the results before choosing a training recipe. Treat 3B vs 8B as a comparison of two released model checkpoints, not a pure parameter-count effect: they are Llama 3.2 and Llama 3.1 respectively.",
        "",
        "## Provenance",
        "",
        f"- TOFU dataset revision: `{report['data_audit']['dataset_revision']}`",
        f"- Open-Unlearning source revision for configs/metric semantics: `{report['open_unlearning_revision']}`",
        "- Dataset audit does not include or download model weights.",
        "",
    ]
    return "\n".join(lines)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path(__file__).resolve().parent / "diagnostics",
        help="Directory for the JSON and Markdown audit reports.",
    )
    args = parser.parse_args()
    report = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "open_unlearning_revision": OPEN_UNLEARNING_REVISION,
        "data_audit": audit_data(),
        "checkpoints": audit_checkpoints(),
        "local_runtime": runtime_context(),
    }
    report["checkpoint_access_checks_pass"] = all(
        item["checkpoint_access_checks_pass"] for item in report["checkpoints"].values()
    )
    args.output_dir.mkdir(parents=True, exist_ok=True)
    json_path = args.output_dir / "tofu_readiness_audit.json"
    md_path = args.output_dir / "tofu_readiness_audit.md"
    json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    md_path.write_text(build_markdown(report), encoding="utf-8")
    print(f"Wrote {json_path}")
    print(f"Wrote {md_path}")
    print(f"Data structural checks: {'PASS' if report['data_audit']['checks']['structural_checks_pass'] else 'FAIL'}")
    print(f"Checkpoint configs available: {sum(item['config_http_access'] == 'PASS' for item in report['checkpoints'].values())}/{len(report['checkpoints'])}")
    print(f"Checkpoint files available: {'PASS' if report['checkpoint_access_checks_pass'] else 'FAIL'}")
    print("Model weights downloaded: 0; training started: no")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/tofu_official_eval.py
#!/usr/bin/env python3
"""Evaluate pinned TOFU checkpoints with the core Open-Unlearning metrics.

This is inference-only. Full checkpoints receive answer likelihood, greedy
generation/ROUGE-L, and truth-ratio evaluation on the canonical TOFU sets.
The paired retain90 oracle checkpoints are scored on forget truth ratios only
to compute the benchmark's forget-quality comparison.
"""

from __future__ import annotations

import argparse
import collections
import hashlib
import json
import math
import random
import shutil
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen

import torch
import torch.nn.functional as F
from rouge_score import rouge_scorer
from scipy.stats import ks_2samp
from transformers import AutoModelForCausalLM, AutoTokenizer


DATASET_ID = "locuslab/TOFU"
DATASET_REVISION = "324592d84ae4f482ac7249b9285c2ecdb53e3a68"
UPSTREAM_REVISION = "2d98027677f3468d52b6baa3fc4f2b9f2dc08820"
PROFILE_QA_COUNT = 20
BOOTSTRAP_SEED = 42
BOOTSTRAP_DRAWS = 5000
MAX_LENGTH = 512
SYSTEM_PROMPT = "You are a helpful assistant."
TEMPLATE_DATE = "10 Apr 2025"
MAX_NEW_TOKENS = 200
IGNORE_INDEX = -100
EPSILON = 1e-10

CHECKPOINTS = {
    "llama_3_2_3b_full": {
        "label": "Llama 3.2 3B full",
        "family": "llama_3_2_3b",
        "role": "full",
        "repo_id": "open-unlearning/tofu_Llama-3.2-3B-Instruct_full",
        "revision": "24f31ca19f6966dcb6f6b29abc511cce71222d4a",
    },
    "llama_3_2_3b_retain90": {
        "label": "Llama 3.2 3B retain90 oracle",
        "family": "llama_3_2_3b",
        "role": "retain90_oracle",
        "repo_id": "open-unlearning/tofu_Llama-3.2-3B-Instruct_retain90",
        "revision": "2c35819e9a731d096df9abd6efbcc361870c5333",
    },
    "llama_3_1_8b_full": {
        "label": "Llama 3.1 8B full",
        "family": "llama_3_1_8b",
        "role": "full",
        "repo_id": "open-unlearning/tofu_Llama-3.1-8B-Instruct_full",
        "revision": "1a5c5b1a557f8c99bdadecd5168ebd03f640b00e",
    },
    "llama_3_1_8b_retain90": {
        "label": "Llama 3.1 8B retain90 oracle",
        "family": "llama_3_1_8b",
        "role": "retain90_oracle",
        "repo_id": "open-unlearning/tofu_Llama-3.1-8B-Instruct_retain90",
        "revision": "f02956a9f25a18f38ad7808fbefb2f957347273e",
    },
}

DATA_FILES = (
    "forget10.json",
    "forget10_perturbed.json",
    "retain90.json",
    "retain_perturbed.json",
    "holdout10.json",
    "real_authors_perturbed.json",
    "world_facts_perturbed.json",
)


def get_bytes(url: str) -> bytes:
    request = Request(url, headers={"User-Agent": "fsds-tofu-official-eval/1.0"})
    with urlopen(request, timeout=60) as response:
        return response.read()


def load_data() -> tuple[dict[str, list[dict]], dict[str, str]]:
    rows_by_file = {}
    hashes = {}
    for filename in DATA_FILES:
        url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/{filename}"
        raw = get_bytes(url)
        rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
        if any(not isinstance(row, dict) for row in rows):
            raise ValueError(f"{filename} contains a non-object JSONL row")
        if any(not isinstance(row.get("question"), str) or not isinstance(row.get("answer"), str) for row in rows):
            raise ValueError(f"{filename} has a missing question/answer string")
        rows_by_file[filename] = rows
        hashes[filename] = hashlib.sha256(raw).hexdigest()
    return rows_by_file, hashes


def build_eval_sets(data: dict[str, list[dict]]) -> dict[str, list[dict]]:
    """Build canonical evaluation rows and attach source profile blocks."""
    retain_source = data["retain90.json"]
    retain_index = collections.defaultdict(list)
    retain_question_answers = collections.defaultdict(set)
    for i, row in enumerate(retain_source):
        retain_index[(row["question"], row["answer"])].append(i)
        retain_question_answers[row["question"]].add(row["answer"])
    collision_indices = {
        i
        for i, row in enumerate(retain_source)
        if len(retain_question_answers[row["question"]]) > 1
    }

    def basic(rows: list[dict], split: str, source_indices: list[int] | None = None) -> list[dict]:
        result = []
        for i, row in enumerate(rows):
            source_i = i if source_indices is None else source_indices[i]
            result.append({
                "split": split,
                "eval_row_index": i,
                "source_row_index": source_i,
                "profile_block": source_i // PROFILE_QA_COUNT if split in {"forget10", "retain90", "holdout10"} else None,
                "question": row["question"],
                "answer": row["answer"],
                "ambiguous_question_collision": split == "retain90" and source_i in collision_indices,
                "paraphrased_answer": row.get("paraphrased_answer"),
                "perturbed_answers": row.get("perturbed_answer"),
            })
        return result

    forget = data["forget10_perturbed.json"]
    forget_source = data["forget10.json"]
    if len(forget) != len(forget_source) or any(
        (a["question"], a["answer"]) != (b["question"], b["answer"])
        for a, b in zip(forget_source, forget)
    ):
        raise ValueError("forget10_perturbed no longer matches forget10 in source order")

    retain = data["retain_perturbed.json"]
    retain_source_indices = []
    for row in retain:
        matches = retain_index.get((row["question"], row["answer"]), [])
        if len(matches) != 1:
            raise ValueError("A canonical retain_perturbed row does not map uniquely to retain90")
        retain_source_indices.append(matches[0])
    if sorted(retain_source_indices) != list(range(20 * PROFILE_QA_COUNT)):
        raise ValueError("Canonical retain_perturbed is not the pinned first-20-profile retain sample")

    return {
        "forget10": basic(forget, "forget10"),
        "retain90": basic(retain, "retain90", retain_source_indices),
        "holdout10": basic(data["holdout10.json"], "holdout10"),
        "real_authors": basic(data["real_authors_perturbed.json"], "real_authors"),
        "world_facts": basic(data["world_facts_perturbed.json"], "world_facts"),
    }


def ids(value) -> list[int]:
    if hasattr(value, "keys") and "input_ids" in value:
        value = value["input_ids"]
    if isinstance(value, torch.Tensor):
        value = value.tolist()
    if value and isinstance(value[0], list):
        value = value[0]
    return [int(x) for x in value]


def encode_answer(tokenizer, question: str, answer: str) -> tuple[list[int], list[int]]:
    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    kwargs = {"date_string": TEMPLATE_DATE}
    full_ids = ids(tokenizer.apply_chat_template(
        conversation, tokenize=True, add_generation_prompt=False, **kwargs
    ))
    prompt_ids = ids(tokenizer.apply_chat_template(
        conversation[:-1], tokenize=True, add_generation_prompt=True, **kwargs
    ))
    if not full_ids or len(prompt_ids) > len(full_ids):
        raise ValueError("Invalid chat-template tokenization")
    if full_ids[: len(prompt_ids)] != prompt_ids:
        raise ValueError("Prompt tokenization is not an exact prefix of the full conversation")
    if tokenizer.eos_token_id is not None and full_ids[-1] != tokenizer.eos_token_id:
        full_ids.append(int(tokenizer.eos_token_id))
    labels = [IGNORE_INDEX] * len(prompt_ids) + full_ids[len(prompt_ids):]
    if len(labels) != len(full_ids) or not any(x != IGNORE_INDEX for x in labels[1:]):
        raise ValueError("Assistant answer mask is empty or misaligned")
    return full_ids, labels


def encode_prompt(tokenizer, question: str) -> list[int]:
    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    result = ids(tokenizer.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        date_string=TEMPLATE_DATE,
    ))
    if not result:
        raise ValueError("Chat-template prompt was empty")
    return result


def collate_scoring(rows: list[tuple[list[int], list[int]]], pad_id: int):
    width = max(len(x[0]) for x in rows)
    input_ids = torch.full((len(rows), width), pad_id, dtype=torch.long)
    attention = torch.zeros((len(rows), width), dtype=torch.long)
    labels = torch.full((len(rows), width), IGNORE_INDEX, dtype=torch.long)
    for i, (tokens, targets) in enumerate(rows):
        n = len(tokens)
        input_ids[i, :n] = torch.tensor(tokens)
        attention[i, :n] = 1
        labels[i, :n] = torch.tensor(targets)
    return input_ids, attention, labels


def score_answers(model, encoded: list[tuple[list[int], list[int]]], pad_id: int, batch_size: int) -> list[dict]:
    device = model.get_input_embeddings().weight.device
    output_scores = []
    offset = 0
    batches = 0
    while offset < len(encoded):
        size = min(batch_size, len(encoded) - offset)
        batch = encoded[offset:offset + size]
        input_ids, attention, labels = collate_scoring(batch, pad_id)
        try:
            with torch.inference_mode():
                out = model(
                    input_ids=input_ids.to(device),
                    attention_mask=attention.to(device),
                    use_cache=False,
                )
                logits = out.logits[:, :-1, :].float()
                targets = labels[:, 1:].to(logits.device)
                losses = F.cross_entropy(
                    logits.reshape(-1, logits.shape[-1]),
                    targets.reshape(-1),
                    ignore_index=IGNORE_INDEX,
                    reduction="none",
                ).reshape(targets.shape)
                tokens_per_row = (targets != IGNORE_INDEX).sum(dim=1)
                if torch.any(tokens_per_row == 0):
                    raise ValueError("Evaluation example has no scored assistant tokens")
                mean_loss = losses.sum(dim=1) / tokens_per_row
                for i in range(size):
                    value = float(mean_loss[i].item())
                    output_scores.append({
                        "avg_answer_token_loss": value,
                        "answer_probability": math.exp(-value),
                        "scored_answer_tokens": int(tokens_per_row[i].item()),
                    })
            offset += size
            batches += 1
            if batches % 100 == 0 or offset == len(encoded):
                print(f"Scored {offset}/{len(encoded)} answer candidates", flush=True)
        except torch.cuda.OutOfMemoryError:
            if size == 1:
                raise
            torch.cuda.empty_cache()
            batch_size = max(1, size // 2)
            print(f"CUDA OOM during scoring; reducing batch size to {batch_size}", flush=True)
    return output_scores


def collate_prompts(rows: list[list[int]], pad_id: int):
    width = max(map(len, rows))
    input_ids = torch.full((len(rows), width), pad_id, dtype=torch.long)
    attention = torch.zeros((len(rows), width), dtype=torch.long)
    for i, row in enumerate(rows):
        input_ids[i, -len(row):] = torch.tensor(row)
        attention[i, -len(row):] = 1
    return input_ids, attention


def generate_answers(model, tokenizer, prompts: list[list[int]], batch_size: int) -> list[str]:
    device = model.get_input_embeddings().weight.device
    generations = []
    offset = 0
    batches = 0
    model.config.use_cache = True
    while offset < len(prompts):
        size = min(batch_size, len(prompts) - offset)
        current = prompts[offset:offset + size]
        input_ids, attention = collate_prompts(current, int(tokenizer.pad_token_id))
        try:
            with torch.inference_mode():
                output = model.generate(
                    input_ids=input_ids.to(device),
                    attention_mask=attention.to(device),
                    do_sample=False,
                    max_new_tokens=MAX_NEW_TOKENS,
                    use_cache=True,
                    pad_token_id=tokenizer.eos_token_id,
                )
            new_tokens = output[:, input_ids.shape[1]:]
            texts = tokenizer.batch_decode(
                new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            generations.extend(text.strip() for text in texts)
            offset += size
            batches += 1
            if batches % 25 == 0 or offset == len(prompts):
                print(f"Generated {offset}/{len(prompts)} answers", flush=True)
        except torch.cuda.OutOfMemoryError:
            if size == 1:
                raise
            torch.cuda.empty_cache()
            batch_size = max(1, size // 2)
            print(f"CUDA OOM during generation; reducing batch size to {batch_size}", flush=True)
    return generations


def build_jobs(eval_sets: dict[str, list[dict]], role: str):
    score_jobs = []
    generation_jobs = []
    for split, rows in eval_sets.items():
        for row in rows:
            key = (split, row["eval_row_index"])
            if role == "full" or split == "forget10":
                score_jobs.append({**row, "variant": "answer", "candidate_answer": row["answer"]})
                if row["paraphrased_answer"]:
                    score_jobs.append({**row, "variant": "paraphrased_answer", "candidate_answer": row["paraphrased_answer"]})
                for i, candidate in enumerate(row["perturbed_answers"] or []):
                    score_jobs.append({**row, "variant": f"perturbed_{i}", "candidate_answer": candidate})
            if role == "full":
                generation_jobs.append(row)
    return score_jobs, generation_jobs


def score_rows_by_key(jobs: list[dict], scores: list[dict]) -> dict[tuple[str, int], dict[str, dict]]:
    result = collections.defaultdict(dict)
    for job, score in zip(jobs, scores):
        key = (job["split"], job["eval_row_index"])
        if job["variant"] in result[key]:
            raise ValueError(f"Duplicate answer candidate score for {key}/{job['variant']}")
        result[key][job["variant"]] = score
    return result


def bootstrap_cluster_ci(rows: list[dict], field: str, seed: int = BOOTSTRAP_SEED) -> list[float] | None:
    groups = collections.defaultdict(list)
    for row in rows:
        if row.get("profile_block") is not None and row.get(field) is not None:
            groups[int(row["profile_block"])].append(float(row[field]))
    if not groups:
        return None
    group_values = list(groups.values())
    rng = random.Random(seed)
    estimates = []
    for _ in range(BOOTSTRAP_DRAWS):
        sampled = [rng.choice(group_values) for _ in group_values]
        values = [value for group in sampled for value in group]
        estimates.append(sum(values) / len(values))
    estimates.sort()
    return [estimates[int(0.025 * (BOOTSTRAP_DRAWS - 1))], estimates[int(0.975 * (BOOTSTRAP_DRAWS - 1))]]


def mean_field(rows: list[dict], field: str) -> float | None:
    values = [float(row[field]) for row in rows if row.get(field) is not None]
    return sum(values) / len(values) if values else None


def summarize_examples(examples: list[dict]) -> dict:
    summary = {"n_questions": len(examples), "n_profile_blocks": len({r["profile_block"] for r in examples if r.get("profile_block") is not None})}
    fields = (
        "answer_probability",
        "rougeL_recall",
        "truth_ratio",
        "truth_ratio_score",
        "normalized_option_probability",
    )
    for field in fields:
        value = mean_field(examples, field)
        if value is not None:
            summary[f"mean_{field}"] = value
            ci = bootstrap_cluster_ci(examples, field)
            if ci is not None:
                summary[f"profile_cluster_bootstrap_95pct_ci_{field}"] = ci
    return summary


def make_metrics(
    rows: list[dict],
    split: str,
    score_map: dict[tuple[str, int], dict[str, dict]],
    generations: dict[tuple[str, int], str],
    tokenizer,
    rouge: rouge_scorer.RougeScorer,
) -> list[dict]:
    metrics = []
    for row in rows:
        key = (split, row["eval_row_index"])
        scores = score_map.get(key, {})
        if "answer" not in scores:
            raise ValueError(f"Missing gold-answer score for {key}")
        output = {
            "split": split,
            "eval_row_index": row["eval_row_index"],
            "source_row_index": row["source_row_index"],
            "profile_block": row["profile_block"],
            "question_sha256": hashlib.sha256(row["question"].encode("utf-8")).hexdigest(),
            "source_answer": row["answer"],
            "reference_answer": row["answer"],
            "ambiguous_question_collision": row["ambiguous_question_collision"],
            "answer_probability": scores["answer"]["answer_probability"],
            "answer_token_loss": scores["answer"]["avg_answer_token_loss"],
            "answer_scored_tokens": scores["answer"]["scored_answer_tokens"],
        }
        if "paraphrased_answer" in scores:
            para = scores["paraphrased_answer"]
            wrong = [scores[f"perturbed_{i}"] for i in range(len(row["perturbed_answers"] or []))]
            output["paraphrased_answer_probability"] = para["answer_probability"]
            output["paraphrased_answer_token_loss"] = para["avg_answer_token_loss"]
            output["perturbed_answer_probabilities"] = [x["answer_probability"] for x in wrong]
            output["perturbed_answer_token_losses"] = [x["avg_answer_token_loss"] for x in wrong]
            if wrong:
                # Match the pinned Open-Unlearning implementation: exponentiate
                # the mean per-candidate answer loss, i.e. the geometric mean of
                # perturbed-answer probabilities.
                geometric_wrong_probability = math.exp(-sum(x["avg_answer_token_loss"] for x in wrong) / len(wrong))
                ratio = geometric_wrong_probability / (para["answer_probability"] + EPSILON)
                output["truth_ratio"] = ratio
                output["truth_ratio_score"] = min(ratio, 1.0 / (ratio + EPSILON)) if split == "forget10" else max(0.0, 1.0 - ratio)
        elif row["perturbed_answers"]:
            wrong = [scores[f"perturbed_{i}"] for i in range(len(row["perturbed_answers"]))]
            wrong_probabilities = [x["answer_probability"] for x in wrong]
            geometric_wrong_probability = math.exp(-sum(x["avg_answer_token_loss"] for x in wrong) / len(wrong))
            ratio = geometric_wrong_probability / (scores["answer"]["answer_probability"] + EPSILON)
            output["perturbed_answer_probabilities"] = wrong_probabilities
            output["perturbed_answer_token_losses"] = [x["avg_answer_token_loss"] for x in wrong]
            output["truth_ratio"] = ratio
            output["truth_ratio_score"] = max(0.0, 1.0 - ratio)
            output["normalized_option_probability"] = scores["answer"]["answer_probability"] / (
                scores["answer"]["answer_probability"] + sum(wrong_probabilities) + EPSILON
            )
        if key in generations:
            text = generations[key]
            full_ids, _ = encode_answer(tokenizer, row["question"], row["answer"])
            prompt_ids = encode_prompt(tokenizer, row["question"])
            input_text = tokenizer.decode(
                prompt_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            full_text = tokenizer.decode(
                full_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            reference = full_text.replace(input_text, "").strip()
            output["generation"] = text
            output["reference_answer"] = reference
            output["rougeL_recall"] = rouge.score(reference, text)["rougeL"].recall
        metrics.append(output)
    return metrics


def model_utility(metrics_by_split: dict[str, list[dict]]) -> dict:
    components = {
        "retain_probability": mean_field(metrics_by_split["retain90"], "answer_probability"),
        "retain_rougeL_recall": mean_field(metrics_by_split["retain90"], "rougeL_recall"),
        "retain_truth_ratio_score": mean_field(metrics_by_split["retain90"], "truth_ratio_score"),
        "real_authors_normalized_option_probability": mean_field(metrics_by_split["real_authors"], "normalized_option_probability"),
        "real_authors_rougeL_recall": mean_field(metrics_by_split["real_authors"], "rougeL_recall"),
        "real_authors_truth_ratio_score": mean_field(metrics_by_split["real_authors"], "truth_ratio_score"),
        "world_facts_normalized_option_probability": mean_field(metrics_by_split["world_facts"], "normalized_option_probability"),
        "world_facts_rougeL_recall": mean_field(metrics_by_split["world_facts"], "rougeL_recall"),
        "world_facts_truth_ratio_score": mean_field(metrics_by_split["world_facts"], "truth_ratio_score"),
    }
    if any(value is None for value in components.values()):
        raise ValueError(f"Cannot compute complete TOFU model utility; missing component: {components}")
    if any(value <= 0 for value in components.values()):
        aggregate = 0.0
    else:
        aggregate = len(components) / sum(1.0 / value for value in components.values())
    return {"components": components, "harmonic_mean": aggregate}


def write_reports(output_dir: Path) -> None:
    keys = ("llama_3_2_3b_full", "llama_3_1_8b_full")
    results = {key: json.loads((output_dir / f"{key}_official_eval.json").read_text(encoding="utf-8")) for key in keys}
    summaries = {key: results[key]["summary"] for key in keys}
    split_labels = {
        "forget10": "Forget10",
        "retain90": "Canonical retain eval",
        "holdout10": "Holdout10",
        "real_authors": "Real authors",
        "world_facts": "World facts",
    }
    lines = [
        "# TOFU official-metric pre-unlearning baseline",
        "",
        "**Status:** complete and inference-only. It evaluates the published full checkpoints and paired retain90 oracle checkpoints. No NPO training or research adapter was run.",
        "",
        "The metric formulas, chat preprocessing, and generated-reference extraction follow the pinned Open-Unlearning implementation. `answer_probability` is mean per-answer-token probability; `rougeL_recall` scores greedy generated answers; Truth Ratio compares the geometric mean likelihood of perturbed answers with the paraphrased correct answer (or source correct answer for real-author/world-fact utility). Kaggle T4 uses FP16 + SDPA here, while upstream configs specify BF16 + FlashAttention 2; formulas and prompts match, but numerical generations are not bitwise-identical to the upstream runtime.",
        "",
        "## Full-checkpoint metrics",
        "",
        "| Split | Model | Questions / profile blocks | Mean answer probability (95% profile CI) | Mean ROUGE-L recall (95% profile CI) | Mean truth ratio | TOFU truth-ratio score | Normalized option probability |",
        "|---|---|---:|---:|---:|---:|---:|---:|",
    ]
    for split, label in split_labels.items():
        for key in keys:
            summary = summaries[key][split]
            def render(metric: str, ci_key: str | None = None) -> str:
                val = summary.get(f"mean_{metric}")
                if val is None:
                    return "—"
                if ci_key and ci_key in summary:
                    low, high = summary[ci_key]
                    return f"{val:.4f} [{low:.4f}, {high:.4f}]"
                return f"{val:.4f}"
            n = f"{summary['n_questions']} / {summary['n_profile_blocks']}" if summary["n_profile_blocks"] else f"{summary['n_questions']} / —"
            lines.append(
                f"| {label} | {results[key]['model']['label']} | {n} | "
                f"{render('answer_probability', 'profile_cluster_bootstrap_95pct_ci_answer_probability')} | "
                f"{render('rougeL_recall', 'profile_cluster_bootstrap_95pct_ci_rougeL_recall')} | "
                f"{render('truth_ratio')} | {render('truth_ratio_score')} | {render('normalized_option_probability')} |"
            )
    lines += ["", "## Composite utility and forget-quality comparison", ""]
    lines.append("| Model | TOFU model-utility harmonic mean | Forget-quality KS p-value vs retain90 oracle |")
    lines.append("|---|---:|---:|")
    for key in keys:
        utility = results[key]["model_utility"]["harmonic_mean"]
        fq = results[key].get("forget_quality_vs_retain90_oracle", {}).get("p_value")
        lines.append(f"| {results[key]['model']['label']} | {utility:.6f} | {fq:.6g}" if fq is not None else f"| {results[key]['model']['label']} | {utility:.6f} | pending oracle result |")

    lines += [
        "",
        "## Retain-question collision sensitivity",
        "",
        "The pinned `retain90` data repeats one generic question with different answers. The canonical 400-row retain evaluation includes one of the two conflicting rows. Primary TOFU metrics retain the official row; the sensitivity row below excludes it.",
        "",
        "| Model | Retain probability, primary → excluding collision | Retain ROUGE-L recall, primary → excluding collision | Retain truth-ratio score, primary → excluding collision |",
        "|---|---:|---:|---:|",
    ]
    for key in keys:
        rows = results[key]["per_example"]["retain90"]
        clean = [row for row in rows if not row["ambiguous_question_collision"]]
        all_summary = summarize_examples(rows)
        clean_summary = summarize_examples(clean)
        lines.append(
            f"| {results[key]['model']['label']} | {all_summary['mean_answer_probability']:.4f} → {clean_summary['mean_answer_probability']:.4f} | "
            f"{all_summary['mean_rougeL_recall']:.4f} → {clean_summary['mean_rougeL_recall']:.4f} | "
            f"{all_summary['mean_truth_ratio_score']:.4f} → {clean_summary['mean_truth_ratio_score']:.4f} |"
        )
    lines += [
        "",
        "## Interpretation limits",
        "",
        "- Confidence intervals resample the 20 inferred profile blocks for forget10, canonical retain, and holdout10. The JSON contains question-level outputs; real-author and world-fact rows have no explicit cluster IDs, so they are reported without profile-cluster intervals.",
        "- Inference uses FP16 + SDPA because Kaggle T4 does not provide the upstream BF16 + FlashAttention 2 runtime. Small numeric and generation differences from an upstream evaluation are possible.",
        "- The 3B and 8B checkpoints come from different Llama generations and training histories. Their paired differences are descriptive checkpoint comparisons, not causal evidence about parameter count.",
        "- A small/large ROUGE or likelihood value alone does not prove forgetting. The forget-quality p-value compares distributions with a retain-only oracle and is not a standalone proof of equivalence or privacy.",
        "- TOFU profiles are fictional. These results cannot establish deletion of real personal data or real-world privacy protection.",
        "- The earlier 6 Nov likelihood-only run used a custom retain sample; this report uses the canonical `retain_perturbed` evaluation set from the pinned benchmark revision.",
        "",
        "## Reproduction metadata",
        "",
        f"Dataset revision: `{DATASET_REVISION}`. Open-Unlearning code revision: `{UPSTREAM_REVISION}`. Bootstrap: {BOOTSTRAP_DRAWS:,} profile-cluster draws, seed {BOOTSTRAP_SEED}. Greedy generation: max {MAX_NEW_TOKENS} new tokens.",
        "",
    ]
    (output_dir / "official_metric_baseline_report.md").write_text("\n".join(lines), encoding="utf-8")

    compare = {}
    for split in split_labels:
        left = results["llama_3_2_3b_full"]["per_example"][split]
        right = results["llama_3_1_8b_full"]["per_example"][split]
        right_by_index = {row["source_row_index"]: row for row in right}
        if len(right_by_index) != len(right) or {row["source_row_index"] for row in left} != set(right_by_index):
            raise ValueError(f"Full checkpoints do not have matched source examples in {split}")
        item = {}
        for metric in ("answer_probability", "rougeL_recall", "truth_ratio", "truth_ratio_score", "normalized_option_probability"):
            pairs = [(row, right_by_index[row["source_row_index"]]) for row in left if metric in row and metric in right_by_index[row["source_row_index"]]]
            if not pairs:
                continue
            deltas = [float(b[metric]) - float(a[metric]) for a, b in pairs]
            metric_result = {"n_questions": len(pairs), "mean_8b_minus_3b": sum(deltas) / len(deltas)}
            grouped = collections.defaultdict(list)
            for (a, _), delta in zip(pairs, deltas):
                if a.get("profile_block") is not None:
                    grouped[int(a["profile_block"])].append(delta)
            if grouped:
                group_ids = list(grouped)
                rng = random.Random(BOOTSTRAP_SEED)
                estimates = []
                for _ in range(BOOTSTRAP_DRAWS):
                    sampled = [rng.choice(group_ids) for _ in group_ids]
                    vals = [value for group in sampled for value in grouped[group]]
                    estimates.append(sum(vals) / len(vals))
                estimates.sort()
                metric_result["profile_cluster_bootstrap_95pct_ci"] = [
                    estimates[int(0.025 * (BOOTSTRAP_DRAWS - 1))],
                    estimates[int(0.975 * (BOOTSTRAP_DRAWS - 1))],
                ]
            else:
                metric_result["interval"] = None
                metric_result["interval_note"] = "No explicit profile cluster IDs; no inferential interval reported."
            item[metric] = metric_result
        compare[split] = item
    compare["interpretation_limit"] = "Descriptive paired checkpoint contrasts only; model generations/training histories differ."
    (output_dir / "paired_full_checkpoint_comparison.json").write_text(json.dumps(compare, indent=2) + "\n", encoding="utf-8")


def encode_jobs(tokenizer, jobs: list[dict]) -> list[tuple[list[int], list[int]]]:
    return [encode_answer(tokenizer, job["question"], job["candidate_answer"]) for job in jobs]


def evaluate_checkpoint(
    key: str,
    output_dir: Path,
    eval_sets: dict[str, list[dict]],
    dataset_hashes: dict[str, str],
    score_batch_size: int,
    generation_batch_size: int,
    max_memory_per_gpu_gib: float,
) -> dict:
    from accelerate import __version__ as accelerate_version
    from transformers import __version__ as transformers_version

    spec = CHECKPOINTS[key]
    model_cache = Path(tempfile.gettempdir()) / "tofu_official_eval_cache" / key
    model_cache.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    tokenizer = AutoTokenizer.from_pretrained(spec["repo_id"], revision=spec["revision"], cache_dir=str(model_cache), use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    if not tokenizer.is_fast:
        raise RuntimeError("Fast tokenizer required")

    score_jobs, generation_jobs = build_jobs(eval_sets, spec["role"])
    encoded_jobs = encode_jobs(tokenizer, score_jobs)
    prompt_ids = [encode_prompt(tokenizer, row["question"]) for row in generation_jobs]
    # Transformers 5.22's allocator warmup reserves a second buffer roughly
    # the size of the 8B model shard. On T4x2 this OOMs before weights load.
    # Disable only this load-time optimization for 8B; checkpoint weights still
    # load normally. Keep the library default for the smaller 3B model.
    allocator_warmup_strategy = "library default"
    if spec["family"] == "llama_3_1_8b":
        import transformers.modeling_utils as modeling_utils

        if not hasattr(modeling_utils, "caching_allocator_warmup"):
            raise RuntimeError("Pinned Transformers version lacks the expected allocator warmup hook")
        modeling_utils.caching_allocator_warmup = lambda *args, **kwargs: None
        allocator_warmup_strategy = "disabled for 8B to avoid duplicate preallocation; weights load normally"

    memory_limit = f"{max_memory_per_gpu_gib:g}GiB"
    # With only two T4s, balanced_low_0 put almost all 8B decoder blocks on
    # GPU 1 and caused CPU offload. Even balancing leaves ample generation
    # headroom on both GPUs at the configured 13 GiB cap.
    print(f"Loading {spec['label']} with balanced; max_memory={memory_limit} per T4", flush=True)
    model = AutoModelForCausalLM.from_pretrained(
        spec["repo_id"],
        revision=spec["revision"],
        cache_dir=str(model_cache),
        dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="balanced",
        max_memory={0: memory_limit, 1: memory_limit, "cpu": "40GiB"},
        attn_implementation="sdpa",
    )
    device_map = getattr(model, "hf_device_map", {})
    offloaded_devices = sorted({str(device) for device in device_map.values() if str(device).lower() in {"cpu", "disk"}})
    print({"loaded_model": spec["label"], "hf_device_map": device_map, "offloaded_devices": offloaded_devices}, flush=True)
    if spec["family"] == "llama_3_1_8b" and offloaded_devices:
        del model, tokenizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise RuntimeError(f"8B placement used CPU/disk offload {offloaded_devices}; refusing a slow, unbounded evaluation")
    model.eval()
    model.config.use_cache = False

    # A tiny non-research prompt tests generation before the full dataset pass.
    smoke_ids = encode_prompt(tokenizer, "Reply with one short greeting.")
    smoke_tensor = torch.tensor([smoke_ids], dtype=torch.long, device=model.get_input_embeddings().weight.device)
    smoke_started = time.monotonic()
    model.config.use_cache = True
    with torch.inference_mode():
        smoke_output = model.generate(
            input_ids=smoke_tensor,
            attention_mask=torch.ones_like(smoke_tensor),
            do_sample=False,
            max_new_tokens=4,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    model.config.use_cache = False
    smoke_new_tokens = int(smoke_output.shape[1] - smoke_tensor.shape[1])
    if smoke_new_tokens < 1:
        raise RuntimeError("GPU placement smoke generated no tokens; refusing full evaluation")
    placement_smoke = {
        "status": "PASS",
        "new_tokens": smoke_new_tokens,
        "elapsed_seconds": round(time.monotonic() - smoke_started, 2),
        "research_rows_used": 0,
    }
    print({"placement_smoke": placement_smoke}, flush=True)

    print(f"Starting answer scoring: {len(encoded_jobs)} candidates", flush=True)
    score_values = score_answers(model, encoded_jobs, int(tokenizer.pad_token_id), score_batch_size)
    if len(score_values) != len(score_jobs):
        raise RuntimeError("Scored row count does not equal job count")
    score_map = score_rows_by_key(score_jobs, score_values)
    generations = {}
    if spec["role"] == "full":
        print(f"Starting greedy generation: {len(prompt_ids)} prompts", flush=True)
        generated_texts = generate_answers(model, tokenizer, prompt_ids, generation_batch_size)
        if len(generated_texts) != len(generation_jobs):
            raise RuntimeError("Generation row count does not equal prompt count")
        generations = {
            (row["split"], row["eval_row_index"]): text
            for row, text in zip(generation_jobs, generated_texts)
        }

    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge.tokenizer = tokenizer
    splits_to_write = ("forget10",) if spec["role"] == "retain90_oracle" else tuple(eval_sets)
    metrics_by_split = {
        split: make_metrics(eval_sets[split], split, score_map, generations, tokenizer, rouge)
        for split in splits_to_write
    }
    result = {
        "status": "PASS_OFFICIAL_TOFU_BASELINE",
        "model": spec,
        "evaluation": {
            "dataset_id": DATASET_ID,
            "dataset_revision": DATASET_REVISION,
            "dataset_sha256_by_file": dataset_hashes,
            "open_unlearning_revision": UPSTREAM_REVISION,
            "profile_block_size": PROFILE_QA_COUNT,
            "bootstrap_seed": BOOTSTRAP_SEED,
            "bootstrap_draws": BOOTSTRAP_DRAWS,
            "generation": {"do_sample": False, "max_new_tokens": MAX_NEW_TOKENS, "use_cache": True},
            "declared_max_input_length": MAX_LENGTH,
            "input_truncation": False,
            "max_scored_sequence_tokens": max((len(tokens) for tokens, _ in encoded_jobs), default=0),
            "scored_sequences_over_declared_max_length": sum(len(tokens) > MAX_LENGTH for tokens, _ in encoded_jobs),
            "max_generation_prompt_tokens": max((len(tokens) for tokens in prompt_ids), default=0),
            "canonical_retain_eval_rows": len(eval_sets["retain90"]),
            "canonical_retain_profile_blocks": sorted({row["profile_block"] for row in eval_sets["retain90"]}),
            "score_jobs": len(score_jobs),
            "generation_jobs": len(generation_jobs),
            "metric_implementation": "Pinned Open-Unlearning probability, ROUGE-L recall, Truth Ratio, normalized option probability, model-utility harmonic mean, and KS forget-quality definitions; see README for exact conventions.",
        },
        "summary": {split: summarize_examples(rows) for split, rows in metrics_by_split.items()},
        "per_example": {split: rows for split, rows in metrics_by_split.items()},
        "model_utility": model_utility(metrics_by_split) if spec["role"] == "full" else None,
        "runtime": {
            "torch_version": torch.__version__,
            "transformers_version": transformers_version,
            "accelerate_version": accelerate_version,
            "rouge_score_version": __import__("importlib.metadata", fromlist=["version"]).version("rouge-score"),
            "scipy_version": __import__("scipy").__version__,
            "inference_dtype": "float16",
            "max_memory_per_gpu_gib": max_memory_per_gpu_gib,
            "device_map_strategy": "balanced",
            "hf_device_map": {str(k): str(v) for k, v in device_map.items()},
            "cpu_offload_used": bool(offloaded_devices),
            "allocator_warmup_strategy": allocator_warmup_strategy,
            "placement_smoke": placement_smoke,
            "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
            "gpu_total_memory_bytes": [torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())],
            "elapsed_seconds": round(time.monotonic() - started, 2),
            "training_started": False,
            "research_adapter_created": False,
        },
    }
    result_path = output_dir / f"{key}_official_eval.json"
    result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    score_path = output_dir / f"{key}_examples.jsonl"
    with score_path.open("w", encoding="utf-8") as f:
        for rows in metrics_by_split.values():
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

    del model, tokenizer, encoded_jobs, prompt_ids
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    shutil.rmtree(model_cache, ignore_errors=True)
    return result


def add_forget_quality(results_dir: Path, full_key: str, oracle_key: str) -> None:
    full_path = results_dir / f"{full_key}_official_eval.json"
    oracle_path = results_dir / f"{oracle_key}_official_eval.json"
    if not full_path.exists() or not oracle_path.exists():
        return
    full = json.loads(full_path.read_text(encoding="utf-8"))
    oracle = json.loads(oracle_path.read_text(encoding="utf-8"))
    full_rows = full["per_example"]["forget10"]
    oracle_rows = oracle["per_example"]["forget10"]
    if [(r["source_row_index"], r["question_sha256"]) for r in full_rows] != [
        (r["source_row_index"], r["question_sha256"]) for r in oracle_rows
    ]:
        raise ValueError("Full and retain90 oracle forget rows are not aligned")
    stat = ks_2samp(
        [1.0 / (row["truth_ratio"] + EPSILON) for row in full_rows],
        [1.0 / (row["truth_ratio"] + EPSILON) for row in oracle_rows],
    )
    full["forget_quality_vs_retain90_oracle"] = {
        "metric": "KS p-value on inverse forget truth-ratio distributions; monotonic inversion leaves the two-sample KS statistic and p-value unchanged",
        "n_full_checkpoint": len(full_rows),
        "n_retain90_oracle": len(oracle_rows),
        "ks_statistic": float(stat.statistic),
        "p_value": float(stat.pvalue),
        "interpretation": "A larger p-value means these two empirical distributions are less distinguishable by this KS test; it is not proof of successful unlearning.",
    }
    full_path.write_text(json.dumps(full, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--output-dir", type=Path, default=Path("/kaggle/working/tofu_official_eval"))
    parser.add_argument("--models", default=",".join(CHECKPOINTS))
    parser.add_argument("--score-batch-size", type=int, default=2)
    parser.add_argument("--generation-batch-size", type=int, default=2)
    parser.add_argument("--max-memory-per-gpu-gib", type=float, default=13.0)
    parser.add_argument("--resume-existing", action="store_true", help="Reuse passing, pinned result files already in output-dir; final verifier checks every row and metric.")
    args = parser.parse_args()
    selected = [x.strip() for x in args.models.split(",") if x.strip()]
    if not selected or set(selected) - set(CHECKPOINTS):
        parser.error(f"models must be selected from {sorted(CHECKPOINTS)}")
    if args.max_memory_per_gpu_gib <= 0 or args.max_memory_per_gpu_gib >= 14:
        parser.error("max-memory-per-gpu-gib must be >0 and <14 to leave T4 runtime headroom")
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        raise RuntimeError("Requires GPU T4 x2; stopped before loading model weights")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    data, hashes = load_data()
    eval_sets = build_eval_sets(data)
    status_path = args.output_dir / "official_eval_status.json"
    reused_results = {}
    if args.resume_existing and status_path.exists():
        previous = json.loads(status_path.read_text(encoding="utf-8"))
        if previous.get("dataset_hashes") != hashes:
            raise RuntimeError("Cannot resume: existing receipt has different pinned dataset hashes")
        for key, entry in previous.get("model_results", {}).items():
            if entry.get("status") != "PASS_OFFICIAL_TOFU_BASELINE":
                continue
            result_path = args.output_dir / f"{key}_official_eval.json"
            examples_path = args.output_dir / f"{key}_examples.jsonl"
            if key not in CHECKPOINTS or not result_path.is_file() or not examples_path.is_file():
                raise RuntimeError(f"Cannot resume: incomplete successful receipt for {key}")
            prior_result = json.loads(result_path.read_text(encoding="utf-8"))
            spec = CHECKPOINTS[key]
            if (
                prior_result.get("status") != "PASS_OFFICIAL_TOFU_BASELINE"
                or prior_result.get("model", {}).get("repo_id") != spec["repo_id"]
                or prior_result.get("model", {}).get("revision") != spec["revision"]
                or prior_result.get("model", {}).get("role") != spec["role"]
                or prior_result.get("evaluation", {}).get("dataset_sha256_by_file") != hashes
            ):
                raise RuntimeError(f"Cannot resume: pinned model/data metadata mismatch for {key}")
            reused_results[key] = entry
        if set(selected) & set(reused_results):
            raise RuntimeError("Requested model is already marked successful; remove it from --models or start a clean output directory")

    for key in selected:
        if CHECKPOINTS[key]["role"] == "full":
            oracle_key = key.replace("_full", "_retain90")
            if oracle_key not in selected and oracle_key not in reused_results:
                parser.error(f"{key} requires its matching retain90 oracle in --models or a verified resumed output")

    status = {
        "status": "RUNNING",
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "selected_models": selected,
        "reused_models": sorted(reused_results),
        "dataset_hashes": hashes,
        "model_results": reused_results,
        "training_started": False,
    }
    status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    for key in selected:
        try:
            result = evaluate_checkpoint(
                key,
                args.output_dir,
                eval_sets,
                hashes,
                args.score_batch_size,
                args.generation_batch_size,
                args.max_memory_per_gpu_gib,
            )
            status["model_results"][key] = {
                "status": result["status"],
                "result_file": f"{key}_official_eval.json",
                "examples_file": f"{key}_examples.jsonl",
                "runtime_seconds": result["runtime"]["elapsed_seconds"],
            }
        except Exception as error:
            shutil.rmtree(Path(tempfile.gettempdir()) / "tofu_official_eval_cache" / key, ignore_errors=True)
            status["status"] = "ERROR"
            status["error"] = f"{type(error).__name__}: {error}"
            status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
            raise
        status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        print(f"{key}: PASS_OFFICIAL_TOFU_BASELINE", flush=True)
    for full_key, oracle_key in (
        ("llama_3_2_3b_full", "llama_3_2_3b_retain90"),
        ("llama_3_1_8b_full", "llama_3_1_8b_retain90"),
    ):
        if full_key in status["model_results"] and oracle_key in status["model_results"]:
            add_forget_quality(args.output_dir, full_key, oracle_key)
    complete_keys = set(CHECKPOINTS)
    if complete_keys <= set(status["model_results"]):
        write_reports(args.output_dir)
    else:
        status["status"] = "PARTIAL"
        status["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
        status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        print(f"Partial results saved under {args.output_dir}; missing {sorted(complete_keys - set(status['model_results']))}")
        return
    status["status"] = "COMPLETE"
    status["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
    status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"Results saved under {args.output_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/verify_tofu_official_eval.py
#!/usr/bin/env python3
"""Independently verify canonical TOFU baseline outputs and derived metrics."""

from __future__ import annotations

import argparse
import collections
import hashlib
import json
import math
import random
from pathlib import Path

from rouge_score import rouge_scorer
from scipy.stats import ks_2samp


EXPECTED = {
    "llama_3_2_3b_full": ("open-unlearning/tofu_Llama-3.2-3B-Instruct_full", "24f31ca19f6966dcb6f6b29abc511cce71222d4a", "full"),
    "llama_3_2_3b_retain90": ("open-unlearning/tofu_Llama-3.2-3B-Instruct_retain90", "2c35819e9a731d096df9abd6efbcc361870c5333", "retain90_oracle"),
    "llama_3_1_8b_full": ("open-unlearning/tofu_Llama-3.1-8B-Instruct_full", "1a5c5b1a557f8c99bdadecd5168ebd03f640b00e", "full"),
    "llama_3_1_8b_retain90": ("open-unlearning/tofu_Llama-3.1-8B-Instruct_retain90", "f02956a9f25a18f38ad7808fbefb2f957347273e", "retain90_oracle"),
}
SPLIT_COUNTS = {
    "forget10": (400, 20),
    "retain90": (400, 20),
    "holdout10": (400, 20),
    "real_authors": (100, None),
    "world_facts": (117, None),
}
EPS = 1e-10
N_BOOTSTRAP = 5000
SEED = 42
EVAL_DATA_FILES = {
    "forget10.json",
    "forget10_perturbed.json",
    "retain90.json",
    "retain_perturbed.json",
    "holdout10.json",
    "real_authors_perturbed.json",
    "world_facts_perturbed.json",
}
METRIC_FIELDS = (
    "answer_probability",
    "rougeL_recall",
    "truth_ratio",
    "truth_ratio_score",
    "normalized_option_probability",
)


def bootstrap_ci(rows: list[dict], field: str) -> list[float] | None:
    grouped = collections.defaultdict(list)
    for row in rows:
        if row.get("profile_block") is not None and row.get(field) is not None:
            grouped[int(row["profile_block"])].append(float(row[field]))
    if not grouped:
        return None
    groups = list(grouped.values())
    rng = random.Random(SEED)
    estimates = []
    for _ in range(N_BOOTSTRAP):
        sample = [rng.choice(groups) for _ in groups]
        values = [x for block in sample for x in block]
        estimates.append(sum(values) / len(values))
    estimates.sort()
    return [estimates[int(.025 * (N_BOOTSTRAP - 1))], estimates[int(.975 * (N_BOOTSTRAP - 1))]]


def mean(rows: list[dict], field: str) -> float | None:
    values = [float(row[field]) for row in rows if row.get(field) is not None]
    return sum(values) / len(values) if values else None


def equal(a, b, field: str, problems: list[str], tolerance: float = 1e-10) -> None:
    if a is None or b is None:
        problems.append(f"{field}: missing numeric value (expected {b}, found {a})")
        return
    if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
        problems.append(f"{field}: nonnumeric value (expected {b!r}, found {a!r})")
        return
    if not math.isfinite(float(a)) or not math.isfinite(float(b)):
        problems.append(f"{field}: non-finite value (expected {b!r}, found {a!r})")
        return
    if not math.isclose(float(a), float(b), rel_tol=tolerance, abs_tol=tolerance):
        problems.append(f"{field}: expected {b}, found {a}")


def verify_rows(key: str, result: dict, audit: dict) -> tuple[list[str], dict]:
    problems = []
    repo, revision, role = EXPECTED[key]
    model = result.get("model", {})
    if result.get("status") != "PASS_OFFICIAL_TOFU_BASELINE":
        problems.append("unexpected result status")
    if (model.get("repo_id"), model.get("revision"), model.get("role")) != (repo, revision, role):
        problems.append("model repository/revision/role does not match pinned protocol")
    runtime = result.get("runtime", {})
    if runtime.get("training_started") is not False or runtime.get("research_adapter_created") is not False:
        problems.append("runtime receipt does not confirm inference-only execution")
    if runtime.get("gpu_names", []).count("Tesla T4") < 2:
        problems.append("receipt does not show two Tesla T4 GPUs")
    if key.startswith("llama_3_1_8b"):
        if runtime.get("device_map_strategy") != "balanced":
            problems.append("8B receipt does not use the pinned balanced device map")
        if runtime.get("cpu_offload_used") is not False:
            problems.append("8B receipt does not confirm GPU-only model placement")
        if runtime.get("placement_smoke", {}).get("status") != "PASS":
            problems.append("8B receipt lacks a passing pre-evaluation generation smoke")
        if not runtime.get("hf_device_map"):
            problems.append("8B receipt lacks the resolved Hugging Face device map")
    if result.get("evaluation", {}).get("input_truncation") is not False:
        problems.append("unexpected input truncation policy")
    expected_files = audit["data_audit"]["files"]
    recorded_hashes = result.get("evaluation", {}).get("dataset_sha256_by_file", {})
    if set(recorded_hashes) != EVAL_DATA_FILES:
        problems.append("dataset checksum file set does not match the pinned evaluation inputs")
    for filename, digest in recorded_hashes.items():
        if filename not in expected_files or expected_files[filename]["sha256"] != digest:
            problems.append(f"dataset checksum mismatch for {filename}")
    if result.get("evaluation", {}).get("open_unlearning_revision") != "2d98027677f3468d52b6baa3fc4f2b9f2dc08820":
        problems.append("Open-Unlearning source revision does not match the pinned protocol")

    expected_splits = ("forget10",) if role == "retain90_oracle" else tuple(SPLIT_COUNTS)
    all_examples = result.get("per_example", {})
    if set(all_examples) != set(expected_splits):
        problems.append(f"unexpected evaluated split keys: {sorted(all_examples)}")
    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    recomputed_summary = {}
    for split in expected_splits:
        rows = all_examples.get(split, [])
        expected_n, expected_groups = SPLIT_COUNTS[split]
        if len(rows) != expected_n:
            problems.append(f"{split}: expected {expected_n} examples, found {len(rows)}")
        source_indices = [row.get("source_row_index") for row in rows]
        if len(set(source_indices)) != len(source_indices):
            problems.append(f"{split}: duplicate source row index")
        if split in {"forget10", "holdout10", "retain90"} and source_indices != list(range(expected_n)):
            problems.append(f"{split}: source row indices do not exactly cover the pinned ordered evaluation rows")
        if split in {"real_authors", "world_facts"} and source_indices != list(range(expected_n)):
            problems.append(f"{split}: source row indices do not exactly cover the pinned ordered utility rows")
        groups = {row.get("profile_block") for row in rows if row.get("profile_block") is not None}
        if expected_groups is not None and len(groups) != expected_groups:
            problems.append(f"{split}: expected {expected_groups} profile groups, found {len(groups)}")
        if expected_groups is not None and groups != set(range(expected_groups)):
            problems.append(f"{split}: profile group IDs are not the expected contiguous blocks")
        for i, row in enumerate(rows):
            p = row.get("answer_probability")
            loss = row.get("answer_token_loss")
            if not isinstance(p, (int, float)) or not math.isfinite(p) or not 0 <= p <= 1:
                problems.append(f"{split} row {i}: invalid answer probability")
            if not isinstance(loss, (int, float)) or not math.isfinite(loss) or loss < 0:
                problems.append(f"{split} row {i}: invalid answer loss")
            else:
                equal(p, math.exp(-float(loss)), f"{split} row {i} exp(-answer loss)", problems)
            if role == "full":
                generation = row.get("generation")
                ref = row.get("reference_answer")
                if not isinstance(generation, str) or not isinstance(ref, str):
                    problems.append(f"{split} row {i}: missing generated text/reference")
                else:
                    expected_rouge = rouge.score(ref, generation)["rougeL"].recall
                    equal(row.get("rougeL_recall"), expected_rouge, f"{split} row {i} ROUGE-L recall", problems)
            if "paraphrased_answer_token_loss" in row:
                para_loss = float(row["paraphrased_answer_token_loss"])
                wrong_losses = row.get("perturbed_answer_token_losses", [])
                if not wrong_losses:
                    problems.append(f"{split} row {i}: truth ratio has no perturbed-answer losses")
                else:
                    wrong_geo = math.exp(-sum(map(float, wrong_losses)) / len(wrong_losses))
                    correct_p = math.exp(-para_loss)
                    ratio = wrong_geo / (correct_p + EPS)
                    equal(row.get("truth_ratio"), ratio, f"{split} row {i} truth ratio", problems)
                    if split == "forget10":
                        score = min(ratio, 1.0 / (ratio + EPS))
                    else:
                        score = max(0.0, 1.0 - ratio)
                    equal(row.get("truth_ratio_score"), score, f"{split} row {i} truth-ratio score", problems)
            elif "perturbed_answer_token_losses" in row:
                wrong_losses = row["perturbed_answer_token_losses"]
                wrong_probs = row.get("perturbed_answer_probabilities", [])
                if len(wrong_losses) != len(wrong_probs) or not wrong_losses:
                    problems.append(f"{split} row {i}: wrong-option losses/probabilities are inconsistent")
                else:
                    for j, (wrong_loss, wrong_prob) in enumerate(zip(wrong_losses, wrong_probs)):
                        equal(wrong_prob, math.exp(-float(wrong_loss)), f"{split} row {i} wrong option {j} probability", problems)
                    wrong_geo = math.exp(-sum(map(float, wrong_losses)) / len(wrong_losses))
                    ratio = wrong_geo / (float(p) + EPS)
                    equal(row.get("truth_ratio"), ratio, f"{split} row {i} truth ratio", problems)
                    equal(row.get("truth_ratio_score"), max(0, 1.0 - ratio), f"{split} row {i} truth-ratio score", problems)
                    normalized = float(p) / (float(p) + sum(map(float, wrong_probs)) + EPS)
                    equal(row.get("normalized_option_probability"), normalized, f"{split} row {i} normalized option probability", problems)
            for field in ("truth_ratio", "truth_ratio_score", "normalized_option_probability", "rougeL_recall"):
                value = row.get(field)
                if value is not None and (not math.isfinite(float(value)) or float(value) < 0):
                    problems.append(f"{split} row {i}: invalid {field}")
                if field == "rougeL_recall" and value is not None and float(value) > 1:
                    problems.append(f"{split} row {i}: ROUGE-L recall is outside [0,1]")

        summary = {"n_questions": len(rows), "n_profile_blocks": len(groups)}
        for field in METRIC_FIELDS:
            value = mean(rows, field)
            if value is None:
                continue
            summary[f"mean_{field}"] = value
            ci = bootstrap_ci(rows, field)
            if ci is not None:
                summary[f"profile_cluster_bootstrap_95pct_ci_{field}"] = ci
        recorded = result.get("summary", {}).get(split, {})
        if summary.keys() != recorded.keys():
            problems.append(f"{split}: summary fields mismatch; recomputed={sorted(summary)}, recorded={sorted(recorded)}")
        for field, value in summary.items():
            if isinstance(value, list):
                if len(recorded.get(field, [])) != len(value):
                    problems.append(f"{split}: {field} length mismatch")
                else:
                    for j, (a, b) in enumerate(zip(recorded[field], value)):
                        equal(a, b, f"{split} {field}[{j}]", problems)
            elif isinstance(value, float):
                equal(recorded.get(field), value, f"{split} {field}", problems)
            elif recorded.get(field) != value:
                problems.append(f"{split}: {field} mismatch")
        recomputed_summary[split] = summary

    if role == "full":
        retain = all_examples["retain90"]
        if [row["source_row_index"] for row in retain] != list(range(400)):
            problems.append("canonical retain evaluation rows are not the pinned first 20 complete profiles")
        collision_rows = [row for row in retain if row.get("ambiguous_question_collision")]
        if len(collision_rows) != 1 or collision_rows[0].get("source_row_index") != 100:
            problems.append("canonical retain evaluation must contain exactly one side of the known collision")
        utility = result.get("model_utility", {})
        component_values = {
            "retain_probability": mean(all_examples["retain90"], "answer_probability"),
            "retain_rougeL_recall": mean(all_examples["retain90"], "rougeL_recall"),
            "retain_truth_ratio_score": mean(all_examples["retain90"], "truth_ratio_score"),
            "real_authors_normalized_option_probability": mean(all_examples["real_authors"], "normalized_option_probability"),
            "real_authors_rougeL_recall": mean(all_examples["real_authors"], "rougeL_recall"),
            "real_authors_truth_ratio_score": mean(all_examples["real_authors"], "truth_ratio_score"),
            "world_facts_normalized_option_probability": mean(all_examples["world_facts"], "normalized_option_probability"),
            "world_facts_rougeL_recall": mean(all_examples["world_facts"], "rougeL_recall"),
            "world_facts_truth_ratio_score": mean(all_examples["world_facts"], "truth_ratio_score"),
        }
        if utility.get("components", {}).keys() != component_values.keys():
            problems.append("model-utility component keys do not match official nine metrics")
        for name, value in component_values.items():
            equal(utility.get("components", {}).get(name), value, f"model utility {name}", problems)
        expected_hm = 0.0 if any(value <= 0 for value in component_values.values()) else 9 / sum(1 / x for x in component_values.values())
        equal(utility.get("harmonic_mean"), expected_hm, "TOFU model-utility harmonic mean", problems)

    return problems, recomputed_summary


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("result_dir", type=Path)
    parser.add_argument("--audit", type=Path, required=True)
    parser.add_argument(
        "--allow-partial",
        action="store_true",
        help="Verify available checkpoints and report PARTIAL when other pinned checkpoints are absent.",
    )
    args = parser.parse_args()
    audit = json.loads(args.audit.read_text(encoding="utf-8"))
    problems = []
    run_status = None
    raw_run_status = None
    run_error = None
    if not audit.get("data_audit", {}).get("checks", {}).get("structural_checks_pass"):
        problems.append("pinned dataset audit did not pass")
    status_file = args.result_dir / "official_eval_status.json"
    if status_file.exists():
        status = json.loads(status_file.read_text(encoding="utf-8"))
        run_status = status.get("status")
        raw_run_status = status.get("raw_kaggle_terminal_status", status.get("raw_run_status"))
        run_error = status.get("error") or status.get("raw_kaggle_error")
        acceptable_statuses = {"COMPLETE", "PARTIAL", "ERROR"} if args.allow_partial else {"COMPLETE"}
        if status.get("status") not in acceptable_statuses:
            problems.append(f"run status is {status.get('status')}, expected COMPLETE")
    else:
        problems.append("missing official_eval_status.json")

    model_results = {}
    recomputed = {}
    for key in EXPECTED:
        path = args.result_dir / f"{key}_official_eval.json"
        if not path.exists():
            model_results[key] = {"status": "MISSING", "problems": [f"missing {path.name}"]}
            if not args.allow_partial:
                problems.append(f"{key}: missing {path.name}")
            continue
        result = json.loads(path.read_text(encoding="utf-8"))
        local_problems, local_summary = verify_rows(key, result, audit)
        model_results[key] = {
            "status": "PASS" if not local_problems else "FAIL",
            "problems": local_problems,
            "model": result.get("model"),
            "summary": local_summary,
            "runtime": result.get("runtime"),
        }
        recomputed[key] = result
        problems.extend(f"{key}: {item}" for item in local_problems)

    for full_key, oracle_key in (
        ("llama_3_2_3b_full", "llama_3_2_3b_retain90"),
        ("llama_3_1_8b_full", "llama_3_1_8b_retain90"),
    ):
        if full_key in recomputed and oracle_key in recomputed:
            full = recomputed[full_key]
            oracle = recomputed[oracle_key]
            full_rows = full["per_example"]["forget10"]
            oracle_rows = oracle["per_example"]["forget10"]
            if [(r["source_row_index"], r["question_sha256"]) for r in full_rows] != [(r["source_row_index"], r["question_sha256"]) for r in oracle_rows]:
                problems.append(f"{full_key}: forget rows do not align with retain oracle")
            else:
                stat = ks_2samp(
                    [1 / (r["truth_ratio"] + EPS) for r in full_rows],
                    [1 / (r["truth_ratio"] + EPS) for r in oracle_rows],
                )
                recorded = full.get("forget_quality_vs_retain90_oracle", {})
                equal(recorded.get("ks_statistic"), stat.statistic, f"{full_key} forget-quality KS statistic", problems)
                equal(recorded.get("p_value"), stat.pvalue, f"{full_key} forget-quality p-value", problems)
    present_statuses = [x["status"] for x in model_results.values() if x["status"] != "MISSING"]
    missing_models = [key for key, value in model_results.items() if value["status"] == "MISSING"]
    if problems or any(value != "PASS" for value in present_statuses):
        overall = "FAIL"
    elif missing_models and args.allow_partial:
        overall = "PARTIAL"
    elif missing_models:
        overall = "FAIL"
    else:
        overall = "PASS"
    verification = {
        "overall_status": overall,
        "run_status": run_status,
        "raw_run_status": raw_run_status,
        "run_error": run_error,
        "audit_status": "PASS" if audit.get("data_audit", {}).get("checks", {}).get("structural_checks_pass") else "FAIL",
        "missing_models": missing_models,
        "problems": problems,
        "models": model_results,
    }
    (args.result_dir / "official_eval_verification.json").write_text(json.dumps(verification, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    lines = ["# TOFU official-metric baseline verification", "", f"Overall: **{overall}**", f"Reconciled run status: **{run_status}**"]
    if raw_run_status:
        lines.append(f"Raw Kaggle terminal status: **{raw_run_status}**")
    if run_error:
        lines.append(f"Raw run error: `{run_error}`")
    lines.append("")
    for key, item in model_results.items():
        lines.append(f"## {key}: {item['status']}")
        lines.append("")
        for message in item.get("problems", []):
            lines.append(f"- {message}")
        for split, summary in item.get("summary", {}).items():
            lines.append(f"- {split}: {summary.get('n_questions')} questions, {summary.get('n_profile_blocks')} profile blocks")
        lines.append("")
    for message in problems:
        if not any(message in item.get("problems", []) for item in model_results.values()):
            lines.append(f"- {message}")
    (args.result_dir / "official_eval_verification.md").write_text("\n".join(lines), encoding="utf-8")
    print(json.dumps(verification, indent=2))
    if overall == "FAIL":
        raise SystemExit(1)
    if overall == "PARTIAL" and not args.allow_partial:
        raise SystemExit(1)


if __name__ == "__main__":
    main()


In [ ]:
from pathlib import Path
import hashlib, json, shutil, zipfile
OUT = Path('/kaggle/working/tofu_official_eval')
manifest_paths = list(Path('/kaggle/input').rglob('partial_resume_manifest.json'))
if len(manifest_paths) != 1: raise RuntimeError(f'Expected exactly one resume manifest in the private dataset; found {manifest_paths}')
resume_root = manifest_paths[0].parent
manifest = json.loads(manifest_paths[0].read_text())
upload_paths = list(Path('/kaggle/input').rglob('upload_manifest.json'))
if len(upload_paths) != 1: raise RuntimeError(f'Expected exactly one upload manifest; found {upload_paths}')
upload_manifest = json.loads(upload_paths[0].read_text())
expected_zip_sha256 = "5882fc3ebf8299413b60503edd9eeee080724909f071f5d3ba34dae683f8ea70"
if upload_manifest.get('archive_sha256') != expected_zip_sha256 or upload_manifest.get('archive_crc') != 'PASS': raise RuntimeError('Uploaded resume-artifact receipt does not match the locally verified archive')
OUT.mkdir(parents=True, exist_ok=True)
for rel, item in manifest['files'].items():
    source = resume_root / rel
    if not source.is_file(): raise RuntimeError(f'Resume dataset is missing {rel}')
    actual = hashlib.sha256(source.read_bytes()).hexdigest()
    if actual != item['sha256']: raise RuntimeError(f'Resume result checksum mismatch: {rel}')
    destination = OUT / rel
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
shutil.copy2(manifest_paths[0], OUT / 'partial_resume_manifest.json')
restore_source = 'private Kaggle dataset; source-file SHA-256 verified (dataset platform unpacked the uploaded ZIP)'
for rel, item in manifest['files'].items():
    source = OUT / rel
    if not source.is_file(): raise RuntimeError(f'Resume archive missing {rel}')
    actual = hashlib.sha256(source.read_bytes()).hexdigest()
    if actual != item['sha256']: raise RuntimeError(f'Restored partial result checksum mismatch: {rel}')
for key in ('llama_3_2_3b_full', 'llama_3_2_3b_retain90', 'llama_3_1_8b_full'):
    result = json.loads((OUT / f'{key}_official_eval.json').read_text())
    examples_path = OUT / f'{key}_examples.jsonl'
    if not examples_path.exists():
        with examples_path.open('w', encoding='utf-8') as f:
            for rows in result['per_example'].values():
                for row in rows: f.write(json.dumps(row, ensure_ascii=False) + '\n')
print({'restore_source': restore_source, 'verified_files': sorted(manifest['files'])})
!python /kaggle/working/diagnose_tofu.py --output-dir /kaggle/working/tofu_official_eval/audit
audit = json.loads((OUT / 'audit/tofu_readiness_audit.json').read_text())
if not audit['data_audit']['checks']['structural_checks_pass'] or not audit.get('checkpoint_access_checks_pass'):
    raise RuntimeError('Pinned data/checkpoint access audit failed; do not evaluate.')

In [ ]:
import subprocess, sys
run = subprocess.run([sys.executable, '/kaggle/working/tofu_official_eval.py', '--output-dir', '/kaggle/working/tofu_official_eval', '--models', 'llama_3_1_8b_retain90', '--score-batch-size', '1', '--generation-batch-size', '1', '--max-memory-per-gpu-gib', '13', '--resume-existing'])
subprocess.run(['bash', '-lc', 'cd /kaggle/working && zip -rq -FS tofu_official_eval_results.zip tofu_official_eval'], check=True)
if run.returncode: raise RuntimeError(f'8B retry failed with status {run.returncode}; partial receipt was archived.')
verify = subprocess.run([sys.executable, '/kaggle/working/verify_tofu_official_eval.py', '/kaggle/working/tofu_official_eval', '--audit', '/kaggle/working/tofu_official_eval/audit/tofu_readiness_audit.json'])
subprocess.run(['bash', '-lc', 'cd /kaggle/working && zip -rq -FS tofu_official_eval_results.zip tofu_official_eval'], check=True)
if verify.returncode: raise RuntimeError(f'Independent verification failed with status {verify.returncode}; download the verification receipt.')
print('All four pinned checkpoint outputs passed independent verification. Download tofu_official_eval_results.zip.')